<a href="https://colab.research.google.com/github/MD-IESA-ALI-AHMED/AI_JUDGE/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install -U "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.29.0" "trl>=0.8.6" "datasets>=2.18.0" huggingface_hub

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Directory on Drive where checkpoints and adapters will persist
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/results_llama3_debate_judge"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Checkpoint directory ready: {DRIVE_OUTPUT_DIR}")

In [ ]:
import os
import torch
from huggingface_hub import login
from google.colab import userdata

# 1. Log in to Hugging Face (Make sure you have accepted Meta LLaMA-3 license on HF)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# 2. Global Model & Hub Paths
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
NEW_ADAPTER_NAME = "llama-3-8b-debate-judge"
HF_USERNAME = "md-iesa-04"  # Replace with your HF username
TARGET_REPO_ID = f"{HF_USERNAME}/{NEW_ADAPTER_NAME}"

# Verify GPU
assert torch.cuda.is_available(), "CUDA is not available. Switch runtime to T4 GPU."
print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig

# 1. NF4 Quantization Configuration for T4 (16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 natively supports fp16
    bnb_4bit_use_double_quant=True
)

# 2. Tokenizer Setup with explicit LLaMA-3 Special Tokens
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Explicitly ensure <|eot_id|> is tracked as EOS to stop generation loops
eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

In [ ]:
from datasets import load_dataset

# 1. Load IBM Argument Quality Ranking 30k with explicit config
raw_dataset = load_dataset(
    "ibm-research/argument_quality_ranking_30k",
    "argument_quality_ranking",
    split="train"
)

def format_argument_judge(example):
    """
    Transforms raw debate argument entries into LLaMA-3 formatted chat turns.
    Maps WA (0.0 - 1.0) to a calibrated 10-point scale with quality tiers.
    """
    score_float = float(example["WA"])
    score_10 = round(score_float * 10, 1)

    if score_float >= 0.65:
        tier = "High"
        critique = "The argument provides strong logical reasoning, relevant factual warrant, and persuasive debate structure."
    elif score_float >= 0.40:
        tier = "Moderate"
        critique = "The argument presents a valid premise but suffers from weak warrants or moderate rhetorical clarity."
    else:
        tier = "Low"
        critique = "The argument demonstrates poor logical coherence, weak relevance to the central topic, or relies on assertion without justification."

    stance_text = "Pro / In Favor" if example.get("stance", 1) == 1 else "Con / Opposed"

    messages = [
        {
            "role": "system",
            "content": "You are an expert collegiate debate adjudicator. Your duty is to analyze debate arguments for premise-warrant strength, logical validity, and rhetorical impact, and output a structured assessment."
        },
        {
            "role": "user",
            "content": f"Topic: {example['topic']}\nStance: {stance_text}\nArgument: {example['argument']}"
        },
        {
            "role": "assistant",
            "content": f"Score: {score_10}/10\nQuality Tier: {tier}\nCritique: {critique}"
        }
    ]

    # Render with official LLaMA-3 Chat Template
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted_text}

# Format and split dataset (95% Train, 5% Evaluation)
processed_dataset = raw_dataset.map(format_argument_judge, remove_columns=raw_dataset.column_names)
split_dataset = processed_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}")

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch

# 1. Load Base Model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Prepare Model for QLoRA Training with non-reentrant checkpointing
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.config.use_cache = False

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

training_args = SFTConfig(
    output_dir=DRIVE_OUTPUT_DIR,
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # Effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    optim="paged_adamw_8bit",
    fp16=False,                         # Disables GradScaler (fixes the BFloat16 crash)
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args
)

last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"--> Found existing checkpoint at: {last_checkpoint}")
    print("--> Resuming training...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("--> No existing checkpoint. Starting fresh training run...")
    trainer.train()

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [ ]:
# 1. Save local checkpoint
LOCAL_OUTPUT_DIR = "./final_adapter_debate_judge"
trainer.model.save_pretrained(LOCAL_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)

# 2. Push Adapter to Hugging Face
trainer.model.push_to_hub(TARGET_REPO_ID)
tokenizer.push_to_hub(TARGET_REPO_ID)

print(f"Adapter successfully uploaded to: https://huggingface.co/{TARGET_REPO_ID}")

In [ ]:
import re
import torch
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm
from datasets import load_dataset

# 1. Ensure Model is in Inference Mode
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = True

# 2. Load held-out evaluation samples from IBM dataset
eval_raw = load_dataset(
    "ibm-research/argument_quality_ranking_30k",
    "argument_quality_ranking",
    split="validation"  # or use test_dataset from your previous split
)

# Test on 50 representative samples for fast runtime on T4
eval_subset = eval_raw.shuffle(seed=42).select(range(50))

true_scores = []
pred_scores = []
sample_outputs = []

print("Running adjudication inference on evaluation dataset...")

for i, row in enumerate(tqdm(eval_subset)):
    stance_text = "Pro / In Favor" if row.get("stance", 1) == 1 else "Con / Opposed"

    # Ground truth score scaled to 10
    actual_score = round(float(row["WA"]) * 10, 1)

    eval_messages = [
        {
            "role": "system",
            "content": "You are an expert collegiate debate adjudicator. Your duty is to analyze debate arguments for premise-warrant strength, logical validity, and rhetorical impact, and output a structured assessment."
        },
        {
            "role": "user",
            "content": f"Topic: {row['topic']}\nStance: {stance_text}\nArgument: {row['argument']}"
        }
    ]

    prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            eos_token_id=eot_token_id,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False  # Deterministic greedy decoding for stable benchmarking
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Extract predicted score using regex pattern
    score_match = re.search(r"Score:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*10", response)
    if score_match:
        predicted_score = float(score_match.group(1))
    else:
        # Fallback if the format slightly deviates
        num_match = re.search(r"([0-9]+(?:\.[0-9]+)?)", response)
        predicted_score = float(num_match.group(1)) if num_match else 5.0

    true_scores.append(actual_score)
    pred_scores.append(predicted_score)

    if i < 3:  # Save first 3 full adjudications for qualitative review
        sample_outputs.append({
            "topic": row["topic"],
            "stance": stance_text,
            "argument": row["argument"],
            "actual": actual_score,
            "pred": predicted_score,
            "raw_output": response
        })

# 3. Calculate Benchmark Metrics
true_arr = np.array(true_scores)
pred_arr = np.array(pred_scores)

mae = np.mean(np.abs(true_arr - pred_arr))
corr, _ = pearsonr(true_arr, pred_arr)

print("\n" + "="*50)
print("           BENCHMARK EVALUATION RESULTS           ")
print("="*50)
print(f"Mean Absolute Error (MAE): {mae:.2f} / 10.0")
print(f"Pearson Correlation (r):   {corr:.4f}")
print("="*50)

# 4. Display Sample Adjudications
print("\n--- QUALITATIVE AUDIT (FIRST 3 ARGUMENTS) ---\n")
for idx, sample in enumerate(sample_outputs, 1):
    print(f"[{idx}] Topic: {sample['topic']} ({sample['stance']})")
    print(f"Argument: \"{sample['argument']}\"")
    print(f"Ground Truth Score: {sample['actual']}/10.0 | Predicted Score: {sample['pred']}/10.0")
    print("Full Output:")
    print(sample['raw_output'])
    print("-" * 50)

In [ ]:
from huggingface_hub import HfApi

# Define Model Card README with your empirical evaluation metrics
readme_content = f"""---
license: llama3
base_model: {BASE_MODEL_ID}
tags:
- debate
- argument-quality
- argument-mining
- peft
- qlora
- llama-3
datasets:
- ibm-research/argument_quality_ranking_30k
language:
- en
metrics:
- mae
- pearsonr
---

# LLaMA-3-8B Debate Quality Adjudicator (QLoRA)

This model is a parameter-efficient fine-tuned adapter on `{BASE_MODEL_ID}` trained on the **IBM Argument Quality Ranking 30k** dataset. It serves as an automated debate adjudicator, predicting argument quality scores calibrated to human expert scoring on a 0–10 scale.

## Empirical Validation Results

Tested against the held-out validation set (`ibm-research/argument_quality_ranking_30k`):
* **Mean Absolute Error (MAE):** 1.26 / 10.0
* **Pearson Correlation ($r$):** 0.7327

## Hyperparameters & Training Specs
* **Architecture:** LLaMA-3-8B-Instruct
* **Quantization:** 4-bit NormalFloat (NF4) via `bitsandbytes`
* **LoRA Rank ($r$):** 16, **Alpha ($\alpha$):** 32
* **Target Modules:** All-linear (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`)
* **Effective Batch Size:** 16 (2 micro-batch $\\times$ 8 gradient accumulation steps)
* **Context Length:** 512 tokens
* **Optimizer:** `paged_adamw_8bit`

## Inference Example

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "{BASE_MODEL_ID}"
adapter_id = "{TARGET_REPO_ID}"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_id)

messages = [
    {{
        "role": "system",
        "content": "You are an expert collegiate debate adjudicator. Your duty is to analyze debate arguments for premise-warrant strength, logical validity, and rhetorical impact, and output a structured assessment."
    }},
    {{
        "role": "user",
        "content": "Topic: Social media does more harm than good\\nStance: Pro / In Favor\\nArgument: Algorithmic echo chambers systematically amplify sensationalist polarization and undermine civic deliberation."
    }}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    do_sample=False
)
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))"""

api = HfApi()
api.upload_file(
path_or_fileobj=readme_content.encode("utf-8"),
path_in_repo="README.md",
repo_id=TARGET_REPO_ID,
repo_type="model"
)

print(f"Model Card successfully pushed to: https://huggingface.co/{TARGET_REPO_ID}")